# Лабораторная работа №2
## Виртуальный датчик концентрации продукта обжига

**Задача.** По минутной телеметрии печи (16 каналов) и нерегулярным лабораторным замерам
прогнозировать концентрацию целевого продукта в текущий момент. Лабораторный результат
поступает с задержкой 10-15 минут, поэтому в признаках используется только история
целевого показателя, но не его текущее значение.

**Данные.**
- `data_train.csv` - минутная телеметрия за 8 месяцев.
- `target_train.csv` - лабораторные замеры (в среднем каждые 2 часа).
- `data_test_small.csv`, `target_test_small.csv` - отложенный тестовый период.

**Структура решения.**
1. EDA и синхронизация источников.
2. Инжиниринг временных признаков.
3. Модели и оценка качества.
4. Выводы и интерпретация.

In [ ]:
%matplotlib inline

import os
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams.update({"figure.max_open_warning": 0, "axes.titlesize": 11})
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 160)

RANDOM_STATE = 42

In [ ]:
# Исходные CSV лежат рядом с ноутбуком, в корне папки lab2.
DATA_DIR = Path(".")
print("DATA_DIR =", DATA_DIR.resolve())

## 2.1. Разведочный анализ и синхронизация данных

Сначала проверяем структуру, пропуски, дубликаты и временные интервалы. Затем совмещаем
телеметрию и лабораторные замеры. Отдельно исследуем задержку: корреляцию целевого
показателя с телеметрией при сдвигах от -120 до +120 минут.

In [ ]:
def read_telemetry(path):
    df = pd.read_csv(path, parse_dates=["datetime"])
    df = df.sort_values("datetime").drop_duplicates("datetime")
    return df.set_index("datetime")

train_raw = read_telemetry(DATA_DIR / "data_train.csv")
test_raw = read_telemetry(DATA_DIR / "data_test_small.csv")

target_train = (
    pd.read_csv(DATA_DIR / "target_train.csv", parse_dates=["Дата"])
    .rename(columns={"Дата": "datetime"})
    .sort_values("datetime")
    .drop_duplicates("datetime")
    .set_index("datetime")
)
target_test = (
    pd.read_csv(DATA_DIR / "target_test_small.csv", parse_dates=["Дата"])
    .rename(columns={"Дата": "datetime"})
    .sort_values("datetime")
    .drop_duplicates("datetime")
    .set_index("datetime")
)

summary = pd.DataFrame({
    "train": [train_raw.shape[0], train_raw.shape[1], train_raw.index.min(), train_raw.index.max()],
    "test": [test_raw.shape[0], test_raw.shape[1], test_raw.index.min(), test_raw.index.max()],
    "target_train": [len(target_train), target_train.shape[1], target_train.index.min(), target_train.index.max()],
    "target_test": [len(target_test), target_test.shape[1], target_test.index.min(), target_test.index.max()],
}, index=["rows", "cols", "start", "end"])
display(summary)
display(target_train.head(3))

In [ ]:
# Доли пропусков по каналам: 12-15 почти пустые, исключаем их из модели.
missing = pd.DataFrame({
    "train_missing": train_raw.isna().mean(),
    "test_missing": test_raw.isna().mean(),
}).sort_values("train_missing", ascending=False)
display((missing * 100).round(3).T)

all_telemetry = [c for c in train_raw.columns if c.startswith("telemetry_")]
sparse_cols = missing.index[missing.train_missing > 0.5].tolist()
telemetry_cols = [c for c in all_telemetry if c not in sparse_cols]
print("Рабочие каналы:", telemetry_cols)
print("Исключены как почти пустые:", sparse_cols)

In [ ]:
# Обзор телеметрии: часовые средние за весь период.
train_raw[telemetry_cols].resample("1h").mean().plot(
    subplots=True, figsize=(15, 11), sharex=True, lw=0.7, legend=False,
    title="Телеметрия, часовые средние"
)
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 8))

axes[0, 0].plot(target_train.index, target_train["target"], lw=0.9, color="tab:blue")
axes[0, 0].plot(target_train.index, target_train["target"].rolling(12).mean(),
                lw=2, color="tab:red", label="скользящее среднее, 12 замеров")
axes[0, 0].set_title("Целевой показатель во времени")
axes[0, 0].legend()

axes[0, 1].hist(target_train["target"], bins=30, color="tab:blue", alpha=0.8)
axes[0, 1].set_title("Распределение целевого показателя")
axes[0, 1].set_xlabel("target")

intervals = target_train.index.to_series().diff().dt.total_seconds().div(60).dropna()
axes[1, 0].hist(intervals, bins=40, color="tab:orange", alpha=0.8)
axes[1, 0].set_title("Интервалы между замерами, минуты")
axes[1, 0].set_xlabel("минуты")

axes[1, 1].plot(target_train.index, target_train["target"], ".", ms=3, alpha=0.6)
axes[1, 1].set_title("Замеры: регулярность и пропуски")
axes[1, 1].set_ylabel("target")

plt.tight_layout()
plt.show()

print("Интервалы, топ-5:")
print(intervals.value_counts().head())
print("\nОписательные статистики target:")
display(target_train["target"].describe().to_frame().T)
print("Уникальных значений target:", target_train["target"].nunique())

In [ ]:
# Оценка задержки: корреляция target с телеметрией при сдвиге lag минут.
def lag_corr(telemetry, target, lag_minutes):
    idx = target.index - pd.Timedelta(minutes=lag_minutes)
    shifted = telemetry.reindex(idx, method="nearest", tolerance=pd.Timedelta("1min"))
    shifted.index = target.index
    return shifted[telemetry_cols].corrwith(target["target"])

lags = np.arange(-120, 121, 10)
corr_by_lag = pd.DataFrame({lag: lag_corr(train_raw, target_train, lag) for lag in lags}).T

fig, ax = plt.subplots(figsize=(12, 6))
sns.heatmap(corr_by_lag.T, cmap="RdBu_r", center=0, ax=ax, cbar_kws={"label": "Pearson r"})
ax.set_title("Корреляция target с телеметрией при сдвиге (lag = target - телеметрия)")
ax.set_xlabel("Сдвиг, минуты")
ax.set_ylabel("Канал")
plt.tight_layout()
plt.show()

best = corr_by_lag.abs().idxmax()
best_corr = corr_by_lag.abs().max()
table = pd.DataFrame({"best_lag_min": best, "max_abs_corr": best_corr}).sort_values(
    "max_abs_corr", ascending=False
)
display(table.round(3))
print("Корреляции при lag=0:")
display(corr_by_lag.loc[0].sort_values(key=np.abs, ascending=False).round(3).to_frame().T)

## 2.2. Инжиниринг признаков для временных рядов

Формируем вектор признаков на каждый момент лабораторного замера:

- сырые значения телеметрии со сдвигами 0, 5, 10, 15, 30, 60 и 120 минут;
- скользящие `mean/std/min/max` по окнам 5-240 минут;
- приращения за 5, 15 и 60 минут;
- календарные признаки (час, день недели, циклические кодировки часа);
- лаги и скользящие статистики самого target, но только по прошлым замерам.

Для теста история target склеивается с train: предыдущий замер к моменту `t` уже известен,
поэтому это не утечка. Текущий замер в признаки не попадает.

In [ ]:
SPARSE_COLS = sparse_cols

def prepare_telemetry(df):
    # Регулярная минутная сетка + короткая интерполяция пропусков.
    out = df.drop(columns=[c for c in df.columns if c in SPARSE_COLS]).copy()
    out = out.asfreq("1min")
    out = out.interpolate(method="time", limit=5).ffill().bfill()
    return out

train_tele = prepare_telemetry(train_raw)
test_tele = prepare_telemetry(test_raw)
print(train_tele.shape, test_tele.shape)
print("Пропуски после подготовки:", int(train_tele.isna().sum().sum()), int(test_tele.isna().sum().sum()))

In [ ]:
SHIFTS = (0, 5, 10, 15, 30, 60, 120)
WINDOWS = (5, 15, 30, 60, 120, 240)


def build_features(telemetry, target_history, eval_index):
    # Признаки телеметрии считаются только из данных до соответствующего момента.
    cols = {}

    for shift in SHIFTS:
        part = telemetry.reindex(
            eval_index - pd.Timedelta(minutes=shift),
            method="nearest",
            tolerance=pd.Timedelta("1min"),
        )
        part.index = eval_index
        for col in telemetry.columns:
            cols[f"{col}_lag{shift}"] = part[col]

    for window in WINDOWS:
        rolling = telemetry.rolling(f"{window}min", min_periods=max(2, window // 5))
        for stat in ("mean", "std", "min", "max"):
            part = getattr(rolling, stat)().reindex(
                eval_index, method="nearest", tolerance=pd.Timedelta("1min")
            )
            part.index = eval_index
            for col in telemetry.columns:
                cols[f"{col}_r{window}_{stat}"] = part[col]

    for shift in (5, 15, 60):
        now = telemetry.reindex(eval_index, method="nearest", tolerance=pd.Timedelta("1min"))
        now.index = eval_index
        before = telemetry.reindex(
            eval_index - pd.Timedelta(minutes=shift),
            method="nearest",
            tolerance=pd.Timedelta("1min"),
        )
        before.index = eval_index
        for col in telemetry.columns:
            cols[f"{col}_d{shift}"] = now[col] - before[col]

    features = pd.DataFrame(cols, index=eval_index)
    features["hour"] = eval_index.hour
    features["dow"] = eval_index.dayofweek
    features["hour_sin"] = np.sin(2 * np.pi * eval_index.hour / 24)
    features["hour_cos"] = np.cos(2 * np.pi * eval_index.hour / 24)

    # История target: только значения до текущего замера.
    target_series = target_history["target"].sort_index()
    for k in (1, 2, 3, 6, 12):
        features[f"tgt_prev{k}"] = target_series.shift(k).reindex(
            eval_index, method="ffill"
        )
    shifted_target = target_series.shift(1)
    for hours in (2, 4, 6, 12, 24):
        features[f"tgt_mean_{hours}h"] = shifted_target.rolling(
            f"{hours}h"
        ).mean().reindex(eval_index, method="ffill")
    features["tgt_diff1"] = features["tgt_prev1"] - features["tgt_prev2"]
    return features


X_train = build_features(train_tele, target_train, target_train.index)
target_history_full = pd.concat([target_train, target_test]).sort_index()
X_test = build_features(test_tele, target_history_full, target_test.index)

y_train = target_train["target"]
y_test = target_test["target"]

# Оставляем только признаки, доступные в train; пропуски заполняем медианой train.
X_train = X_train.loc[:, X_train.notna().any()]
X_test = X_test[X_train.columns]
train_medians = X_train.median()
X_train = X_train.fillna(train_medians)
X_test = X_test.fillna(train_medians)

print("X_train:", X_train.shape, "| X_test:", X_test.shape)
print("Признаков target-истории:", sum(c.startswith("tgt_") for c in X_train.columns))
display(X_train[["tgt_prev1", "tgt_mean_4h", "telemetry_0_lag0"]].head(3))

In [ ]:
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
plot_acf(target_train["target"], lags=24, ax=axes[0], zero=False)
axes[0].set_title("ACF target")
plot_pacf(target_train["target"], lags=24, ax=axes[1], zero=False, method="ywm")
axes[1].set_title("PACF target")
plt.tight_layout()
plt.show()

print("Автокорреляция target:")
for lag in (1, 2, 3, 6, 12):
    print(f"  lag {lag:2d} наблюдений: {target_train['target'].autocorr(lag):+.3f}")

In [ ]:
# Отбор признаков: F-тест и корреляция с target.
from sklearn.feature_selection import f_regression

f_scores, _ = f_regression(X_train, y_train)
f_table = pd.Series(f_scores, index=X_train.columns).sort_values(ascending=False)

corr_table = X_train.corrwith(y_train).abs().sort_values(ascending=False)

print("Топ-20 признаков по F-тесту:")
display(f_table.head(20).round(2).to_frame("F"))
print("Топ-20 признаков по |corr|:")
display(corr_table.head(20).round(3).to_frame("|corr|"))

In [ ]:
# PCA: проверяем, сколько компонент нужно для сжатия пространства признаков.
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

scaler_pca = StandardScaler()
X_scaled = scaler_pca.fit_transform(X_train)
pca = PCA(random_state=RANDOM_STATE).fit(X_scaled)
explained = np.cumsum(pca.explained_variance_ratio_)

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(np.arange(1, len(explained) + 1), explained, marker=".")
ax.axhline(0.95, color="tab:red", ls="--", lw=1)
ax.set_xlabel("Число компонент")
ax.set_ylabel("Накопленная объяснённая дисперсия")
ax.set_title("PCA на признаках")
plt.tight_layout()
plt.show()

print("Компонент для 90% дисперсии:", int(np.searchsorted(explained, 0.90) + 1))
print("Компонент для 95% дисперсии:", int(np.searchsorted(explained, 0.95) + 1))

## 2.3. Прогнозные модели

Рассматриваем три класса моделей:

1. **Ridge-регрессия** - линейная модель с L2-регуляризацией для большого числа
   коррелирующих признаков; хорошо интерпретируется.
2. **ExtraTrees** - нелинейный ансамбль деревьев, устойчив к масштабу и выбросам,
   моделирует взаимодействия признаков.
3. **HistGradientBoosting** - градиентный бустинг для табличных данных, эффективен
   на средних выборках, поддерживает пропуски и нелинейности.

Дополнительно считаем две базовые стратегии: последний известный замер (persistence)
и скользящее среднее последних замеров.

**Что точно не подходит.**
- ARIMA/SARIMA в чистом виде: target нерегулярен, прогноз строится на одной переменной без
  телеметрии; экзогенные высокочастотные каналы теряются.
- Prophet: рассчитан на регулярный ряд и тренд-сезонность, а не на краткосрочную
  динамику процесса по 12 датчикам.
- LSTM/TCN/Transformer: всего 1773 размеченных точки, глубокие модели будут переобучаться,
  требуют много вычислений и хуже интерпретируются.
- VAR/VECM: требует регулярного совместного ряда, а target наблюдается только раз в 2 часа.

In [ ]:
from sklearn.base import clone
from sklearn.ensemble import ExtraTreesRegressor, HistGradientBoostingRegressor
from sklearn.feature_selection import SelectKBest, f_regression
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, mean_absolute_percentage_error
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler


def wape(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    return np.abs(y_true - y_pred).sum() / np.abs(y_true).sum()


def regression_metrics(y_true, y_pred):
    return {
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
        "MAPE, %": mean_absolute_percentage_error(y_true, y_pred) * 100,
        "WAPE, %": wape(y_true, y_pred) * 100,
    }

In [ ]:
# Подбор k и alpha для Ridge по TimeSeriesSplit только на train.
ridge_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("select", SelectKBest(score_func=f_regression)),
    ("scaler", StandardScaler()),
    ("ridge", Ridge()),
])

param_grid = {
    "select__k": [40, 60, 80, 120],
    "ridge__alpha": [10, 30, 100, 300],
}
tscv = TimeSeriesSplit(n_splits=5)
grid = GridSearchCV(
    ridge_pipe,
    param_grid=param_grid,
    cv=tscv,
    scoring="neg_mean_absolute_error",
    n_jobs=-1,
)
grid.fit(X_train, y_train)

best_ridge = grid.best_estimator_
selected_mask = best_ridge.named_steps["select"].get_support()
selected_cols = X_train.columns[selected_mask]
print("Лучшие параметры Ridge:", grid.best_params_)
print("CV MAE:", round(-grid.best_score_, 4))
print("Отобрано признаков:", len(selected_cols))

In [ ]:
X_train_sel = X_train[selected_cols]
X_test_sel = X_test[selected_cols]

extra_trees = ExtraTreesRegressor(
    n_estimators=600,
    max_depth=12,
    min_samples_leaf=3,
    max_features=0.4,
    n_jobs=-1,
    random_state=RANDOM_STATE,
)
hgb = HistGradientBoostingRegressor(
    loss="absolute_error",
    max_iter=600,
    learning_rate=0.03,
    max_leaf_nodes=15,
    min_samples_leaf=20,
    l2_regularization=5.0,
    random_state=RANDOM_STATE,
)

models = {
    "Ridge (CV)": best_ridge,
    "ExtraTrees": extra_trees,
    "HistGradientBoosting": hgb,
}

predictions = {}
timings = {}
for name, model in models.items():
    t0 = time.perf_counter()
    model.fit(X_train_sel, y_train)
    predictions[name] = model.predict(X_test_sel)
    timings[name] = time.perf_counter() - t0

# Базовые стратегии.
predictions["Persistence"] = X_test["tgt_prev1"].to_numpy()
predictions["Скользящее среднее"] = X_test["tgt_mean_4h"].to_numpy()
timings["Persistence"] = 0.0
timings["Скользящее среднее"] = 0.0

rows = []
for name, pred in predictions.items():
    row = regression_metrics(y_test, pred)
    row["fit, сек"] = timings[name]
    rows.append(pd.Series(row, name=name))
metrics_df = pd.DataFrame(rows)
display(metrics_df.round(4).sort_values("WAPE, %"))

In [ ]:
# Направление изменения: сравниваем знак приращения относительно предыдущего замера.
def direction_accuracy(y_true, y_pred, previous):
    actual_delta = np.asarray(y_true) - np.asarray(previous)
    pred_delta = np.asarray(y_pred) - np.asarray(previous)
    mask = np.abs(actual_delta) > 1e-12
    return np.mean(np.sign(actual_delta[mask]) == np.sign(pred_delta[mask]))


previous = X_test["tgt_prev1"].to_numpy()
direction = {
    name: direction_accuracy(y_test.to_numpy(), pred, previous)
    for name, pred in predictions.items()
}
display(pd.Series(direction, name="Direction accuracy").sort_values(ascending=False).to_frame().round(3))

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(15, 9), sharex=False)
best_name = metrics_df["WAPE, %"].idxmin()
best_pred = predictions[best_name]

axes[0].plot(y_test.index, y_test.values, color="black", lw=1.2, label="Факт")
axes[0].plot(y_test.index, predictions["Ridge (CV)"], color="tab:blue", lw=1.0,
             label="Ridge")
axes[0].plot(y_test.index, predictions["HistGradientBoosting"], color="tab:orange",
             lw=0.9, alpha=0.8, label="HistGB")
axes[0].set_title("Прогноз на тестовом периоде")
axes[0].set_ylabel("target")
axes[0].legend(ncol=3)

zoom = y_test.index < (y_test.index.min() + pd.Timedelta(days=14))
axes[1].plot(y_test.index[zoom], y_test.values[zoom], "o-", ms=3, color="black",
             lw=1, label="Факт")
axes[1].plot(y_test.index[zoom], best_pred[zoom], "o-", ms=3, color="tab:red",
             lw=1, label=f"Лучшая модель: {best_name}")
axes[1].set_title("Первые две недели теста")
axes[1].set_ylabel("target")
axes[1].legend()

plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(y_test, best_pred, s=14, alpha=0.55, label=best_name)
lims = [min(y_test.min(), best_pred.min()), max(y_test.max(), best_pred.max())]
ax.plot(lims, lims, "k--", lw=1, label="идеальный прогноз")
ax.set_xlabel("Факт")
ax.set_ylabel("Прогноз")
ax.set_title("Диаграмма рассеяния на тесте")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Анализ остатков лучшей модели на тесте.
from scipy import stats
from statsmodels.stats.diagnostic import het_breuschpagan, acorr_ljungbox
from statsmodels.stats.stattools import jarque_bera

residuals = y_test.to_numpy() - best_pred

jb_stat, jb_p, _, _ = jarque_bera(residuals)
lb = acorr_ljungbox(residuals, lags=[6, 12], return_df=True)
bp = het_breuschpagan(residuals, np.column_stack([np.ones(len(best_pred)), best_pred]))

diag = pd.DataFrame({
    "тест": ["Jarque-Bera (нормальность)", "Ljung-Box lag=6", "Ljung-Box lag=12",
             "Breusch-Pagan (гетероскедастичность)"],
    "статистика": [jb_stat, lb.loc[6, "lb_stat"], lb.loc[12, "lb_stat"], bp[0]],
    "p-value": [jb_p, lb.loc[6, "lb_pvalue"], lb.loc[12, "lb_pvalue"], bp[1]],
})
display(diag.round(4))

fig, axes = plt.subplots(2, 2, figsize=(14, 8))
axes[0, 0].hist(residuals, bins=30, color="tab:blue", alpha=0.8)
axes[0, 0].set_title("Распределение остатков")

stats.probplot(residuals, dist="norm", plot=axes[0, 1])
axes[0, 1].set_title("Q-Q остатков")

plot_acf(residuals, lags=24, ax=axes[1, 0], zero=False)
axes[1, 0].set_title("ACF остатков")

axes[1, 1].scatter(best_pred, residuals, s=12, alpha=0.6)
axes[1, 1].axhline(0, color="black", lw=1)
axes[1, 1].set_title("Остатки против прогноза")
axes[1, 1].set_xlabel("Прогноз")

plt.tight_layout()
plt.show()

In [ ]:
# Важность признаков: permutation importance и SHAP для ExtraTrees.
from sklearn.inspection import permutation_importance

perm = permutation_importance(
    best_ridge, X_test_sel, y_test,
    scoring="neg_mean_absolute_error",
    n_repeats=12, random_state=RANDOM_STATE, n_jobs=-1,
)
perm_series = pd.Series(perm.importances_mean, index=selected_cols).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(9, 5))
perm_series.head(15).iloc[::-1].plot.barh(ax=ax, color="tab:green")
ax.set_title("Permutation importance, Ridge")
ax.set_xlabel("рост MAE при перемешивании")
plt.tight_layout()
plt.show()

print("Топ-15 по permutation importance:")
display(perm_series.head(15).to_frame("importance").round(4))

In [ ]:
import shap

explainer = shap.TreeExplainer(extra_trees)
shap_values = explainer.shap_values(X_test_sel)
if isinstance(shap_values, list):
    shap_values = shap_values[0]

shap.summary_plot(shap_values, X_test_sel, max_display=15, show=False)
plt.title("SHAP для ExtraTrees")
plt.tight_layout()
plt.show()

shap_importance = pd.Series(
    np.abs(shap_values).mean(axis=0), index=selected_cols
).sort_values(ascending=False)
print("Топ-15 по |SHAP|:")
display(shap_importance.head(15).to_frame("mean |SHAP|").round(4))

In [ ]:
# AIC/BIC для линейных OLS-моделей с разным числом признаков + оценка на тесте.
import statsmodels.api as sm

ic_rows = []
for k in (20, 40, 80, 120):
    selector = SelectKBest(score_func=f_regression, k=k).fit(X_train, y_train)
    cols_k = X_train.columns[selector.get_support()]
    X_tr = sm.add_constant(X_train[cols_k].astype(float))
    X_te = sm.add_constant(X_test[cols_k].astype(float), has_constant="add")
    ols = sm.OLS(y_train.astype(float), X_tr).fit()
    pred = ols.predict(X_te)
    ic_rows.append({
        "k признаков": k,
        "AIC": ols.aic,
        "BIC": ols.bic,
        "MAE test": mean_absolute_error(y_test, pred),
        "WAPE test, %": wape(y_test, pred) * 100,
    })

ic_df = pd.DataFrame(ic_rows).set_index("k признаков")
display(ic_df.round(4))

In [ ]:
# Итоговая таблица и сохранение прогнозов лучшей модели.
final = metrics_df.copy()
final["Direction acc."] = pd.Series(direction)
display(final.round(4).sort_values("WAPE, %"))

prediction_table = pd.DataFrame({
    "datetime": y_test.index,
    "target": y_test.to_numpy(),
    "prediction_best": best_pred,
    "prediction_ridge": predictions["Ridge (CV)"],
    "prediction_hgb": predictions["HistGradientBoosting"],
    "prediction_extra_trees": predictions["ExtraTrees"],
})
prediction_table.to_csv("predictions_test.csv", index=False, date_format="%Y-%m-%d %H:%M:%S")
print("Лучшая модель по WAPE:", best_name)
print("Файл predictions_test.csv сохранён.")